# Anlu Health MedGemma 1.5 QLoRA (Colab)

This notebook trains a small behavioral adapter on a separate GPU runtime. It does not turn the model into a diagnostic system. Accept the Health AI Developer Foundations terms on Hugging Face, use only reviewed synthetic/de-identified data, and run the repository safety gates before any promotion. A T4/L4-class GPU is recommended.

In [ ]:
!pip -q install 'transformers>=4.53,<5' 'datasets>=4,<5' 'peft>=0.16,<1' 'trl>=0.19,<1' 'bitsandbytes>=0.46,<1' 'accelerate>=1.9,<2' 'mlflow>=3.1,<4'

## Authenticate and load validated data
Set `HF_TOKEN` in Colab Secrets. Clone your private repository or upload only the validated JSONL artifact. Never upload user conversations.

In [ ]:
import os, json, hashlib, torch
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive
from datasets import load_dataset
from huggingface_hub import login
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive/Anlu Health')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = DRIVE_ROOT / 'Model Adapters' / RUN_ID
EVAL_DIR = DRIVE_ROOT / 'Evaluation Reports' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
EVAL_DIR.mkdir(parents=True, exist_ok=False)
login(token=os.environ['HF_TOKEN'])
BASE_MODEL = 'google/medgemma-1.5-4b-it'
DATA_PATH = os.environ.get('ANLU_DATA_PATH', str(DRIVE_ROOT / 'Datasets' / 'train.validated.jsonl'))
dataset = load_dataset('json', data_files=DATA_PATH, split='train').train_test_split(test_size=0.2, seed=42)
print(dataset)

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
processor = AutoProcessor.from_pretrained(BASE_MODEL)
model = AutoModelForImageTextToText.from_pretrained(BASE_MODEL, quantization_config=quant, device_map='auto', torch_dtype=torch.bfloat16)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])

In [ ]:
def format_example(example):
    return processor.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)

args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    max_length=2048,
    seed=42,
    report_to='none',
)
trainer = SFTTrainer(model=model, args=args, train_dataset=dataset['train'], eval_dataset=dataset['test'], peft_config=lora, processing_class=processor, formatting_func=format_example)

In [ ]:
result = trainer.train()
metrics = trainer.evaluate()
print(metrics)
# Loss is not a medical-safety metric. Download the adapter only after the separate held-out gates and clinician review pass.

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / 'adapter-candidate'
trainer.model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
data_sha = hashlib.sha256(Path(DATA_PATH).read_bytes()).hexdigest()
manifest = {'status':'candidate','base_model':BASE_MODEL,'data_sha256':data_sha,'seed':42,'metrics':metrics}
(ADAPTER_DIR / 'candidate_manifest.json').write_text(json.dumps(manifest, indent=2))
(EVAL_DIR / 'training_metrics.json').write_text(json.dumps(metrics, indent=2, default=str))
print(json.dumps(manifest, indent=2))

## Required next step
Run the adapter behind the same OpenAI-compatible endpoint used by the app, execute the held-out multilingual safety and RAG evaluations, record adapter hashes in `model_registry/production.yaml`, and obtain all named human approvals. Do not promote directly from this notebook.